In [2]:
!pip install pyspark

  Using cached pyspark-4.1.1-py2.py3-none-any.whl
  Using cached py4j-0.10.9.9-py2.py3-none-any.whl.metadata (1.3 kB)
Using cached py4j-0.10.9.9-py2.py3-none-any.whl (203 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pyspark]m1/2 [pyspark]


# Repartition

In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("homework-q2") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/07 17:53:48 WARN Utils: Your hostname, JMB-PC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/07 17:53:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/07 17:53:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [9]:
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")

In [5]:
df.printSchema()
df.show(5)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+------

In [6]:
df_repart = df.repartition(4)

In [7]:
df_repart.rdd.getNumPartitions()

4

In [8]:
output_path = "yellow_tripdata_2025-11-repartitioned"

df_repart.write.mode("overwrite").parquet(output_path)

# Count Records

In [11]:
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [14]:
df.createOrReplaceTempView("taxi_trips")

spark.sql("""
SELECT COUNT(*)
FROM taxi_trips
WHERE tpep_pickup_datetime >= '2025-11-15'
AND tpep_pickup_datetime < '2025-11-16'
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



# Longest trip

In [1]:
from pyspark.sql.functions import col

In [6]:
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")

In [13]:
from pyspark.sql.functions import max, unix_timestamp

longest_trip = df.select(
    (
        (
            unix_timestamp("tpep_dropoff_datetime") -
            unix_timestamp("tpep_pickup_datetime")
        )   / 3600 
    )
    .alias("trip_hours") 
)

longest_trip.agg(max("trip_hours") ).show()

+-----------------+
|  max(trip_hours)|
+-----------------+
|90.64666666666666|
+-----------------+



# Least frequent pickup location zone

In [14]:
zones = spark.read.option("header", True).csv("taxi_zone_lookup.csv")

In [15]:
zones.show(5)
zones.printSchema()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows
root
 |-- LocationID: string (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [16]:
zones.createOrReplaceTempView("zones")
df.createOrReplaceTempView("trips")

In [17]:
spark.sql("""
SELECT
    z.Zone,
    COUNT(*) as trip_count
FROM trips t
JOIN zones z
ON t.PULocationID = z.LocationID
GROUP BY z.Zone
ORDER BY trip_count ASC
LIMIT 1
""").show()

+--------------------+----------+
|                Zone|trip_count|
+--------------------+----------+
|Governor's Island...|         1|
+--------------------+----------+



In [18]:
from pyspark.sql.functions import count

df.join(
    zones,
    df.PULocationID == zones.LocationID
).groupBy(
    "Zone"
).agg(
    count("*").alias("trip_count")
).orderBy(
    "trip_count"
).show(1)

+--------------------+----------+
|                Zone|trip_count|
+--------------------+----------+
|Governor's Island...|         1|
+--------------------+----------+
only showing top 1 row
